In [4]:
# 1. Clone your collaboration repository
# Replace with your actual GitHub URL
!git clone https://github.com/nmorok/Teleconnections-ViT.git

# 2. Enter the repository directory
import os
os.chdir('Teleconnections-ViT')
# 3. Add the current directory to sys.path so 'import model' works
import sys
sys.path.append(os.getcwd())

# 4. Verify the files are present
print("Files in current directory:", os.listdir())

Cloning into 'Teleconnections-ViT'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 95 (delta 27), reused 86 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 14.86 MiB | 9.35 MiB/s, done.
Resolving deltas: 100% (27/27), done.
Files in current directory: ['README.md', '.git', '.gitignore', 'data', 'LICENSE', 'requirements.txt', 'models', 'notebook', 'Pipeline.md']


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import torch

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define your data path (Update this to your actual Drive folder)
# Usually formatted as: /content/drive/MyDrive/Folder_Name
DATA_PATH = '/content/drive/MyDrive/Teleconnection_ViT'

# 3. Quick verification check
if os.path.exists(DATA_PATH):
    print(f"✓ Data folder found at: {DATA_PATH}")
    print("Files available:", os.listdir(DATA_PATH))
else:
    print(f"✗ ERROR: Could not find folder at {DATA_PATH}. Check your Drive path.")

# 4. Hardware Check
# 1. Check if CUDA (GPU support) is available
print(f"Is CUDA available? {torch.cuda.is_available()}")

# 2. Get the name of the GPU assigned by Colab Pro
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    # Correct way to check memory
    props = torch.cuda.get_device_properties(0)
    print(f"Total GPU Memory: {props.total_memory / 1e9:.2f} GB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Data folder found at: /content/drive/MyDrive/Teleconnection_ViT
Files available: ['train_spawners.npy', 'val_recruits.npy', 'train_recruits.npy', 'val_spawners.npy', 'test_spawners.npy', 'test_recruits.npy']
Is CUDA available? True
GPU Name: NVIDIA A100-SXM4-80GB
Total GPU Memory: 85.17 GB


In [ ]:
from google.colab import output
import time

def keep_alive():
    while True:
        time.sleep(300)  # Sleep for 5 minutes
        output.clear()
        print("Session active...")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from model import CrabTransformer
from losses import TweedieLoss
from data.data_helper import CrabDataset

In [ ]:
def train_model():

    # set hyperparameters:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")
    batch_size = 100
    epochs = 50
    learning_rate = 1e-4
    grid_size = 50
    patch_size = 5
    in_channels = 2
    embed_dim = 128
    num_heads = 8
    d_ff = 512
    num_layers = 6
    mask = False
    dropout = 0.1

    # Create checkpoint directory in Google Drive
    CHECKPOINT_DIR = os.path.join(DATA_PATH, "checkpoints")
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

    # prepare data
    train_ds = CrabDataset(
        spawner_path=os.path.join(DATA_PATH, "train_spawners.npy"),
        recruit_path=os.path.join(DATA_PATH, "train_recruits.npy"),
        n_years=30
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)

    val_ds = CrabDataset(
        spawner_path=os.path.join(DATA_PATH, "val_spawners.npy"),
        recruit_path=os.path.join(DATA_PATH, "val_recruits.npy"),
        n_years=30
    )

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    # initialize the model, loss, and optimizer
    model = CrabTransformer(grid_size, patch_size, in_channels, embed_dim, num_heads, num_layers, d_ff, mask, dropout).to(device)
    criterion = TweedieLoss(power=1.5)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # training loop
    model.train()
    for epoch in range(epochs):
        total_train_loss = 0
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            # 'data' is already the [Batch, 2, 10, 10] tensor
            # 'target' is the [Batch, 1, 10, 10] recruitment density
            inputs, targets = inputs.to(device), targets.to(device)

            # clear the gradients
            optimizer.zero_grad()
            # forward pass
            outputs = model(inputs)
            # calculate loss
            loss = criterion(outputs, targets)
            # backward pass (calculate gradients)
            loss.backward()
            # update weights
            optimizer.step()

            total_train_loss += loss.item()

        # --- VALIDATION PHASE ---
        model.eval() # Set model to evaluation mode
        total_val_loss = 0
        with torch.no_grad(): # Disable gradient calculation (saves memory/time)
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                total_val_loss += loss.item()

        avg_train = total_train_loss / len(train_loader)
        avg_val = total_val_loss / len(val_loader)

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f}")


        # --- CHECKPOINT SAVING ---
        # Save checkpoint every 5 epochs
        if (epoch + 1) % 5 == 0:
            checkpoint_path = os.path.join(CHECKPOINT_DIR, f"checkpoint_epoch_{epoch+1}.pt")
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train,
                'val_loss': avg_val,
            }, checkpoint_path)
            print(f"✓ Checkpoint saved: {checkpoint_path}")

        # Save best model
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            best_model_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': avg_train,
                'val_loss': avg_val,
            }, best_model_path)
            print(f"✓ New best model saved! Val Loss: {avg_val:.4f}")

    # Save final model
    final_model_path = os.path.join(CHECKPOINT_DIR, "final_model.pt")
    torch.save({
        'epoch': epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': avg_train,
        'val_loss': avg_val,
    }, final_model_path)
    print(f"✓ Final model saved: {final_model_path}")

    return model


def load_checkpoint(checkpoint_path, model, optimizer=None):
    """
    Load a saved checkpoint.

    Args:
        checkpoint_path: Path to the .pt file
        model: Model instance to load weights into
        optimizer: (Optional) Optimizer to load state into

    Returns:
        epoch: Epoch number from checkpoint
    """
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])

    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    epoch = checkpoint['epoch']
    train_loss = checkpoint['train_loss']
    val_loss = checkpoint['val_loss']

    print(f"Loaded checkpoint from epoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    return epoch

In [ ]:
if __name__ == "__main__":
    mod1 = train_model()

In [ ]:
# View recent checkpoints
import glob

checkpoint_dir = os.path.join(DATA_PATH, "checkpoints")
checkpoints = glob.glob(os.path.join(checkpoint_dir, "*.pt"))

print("Available checkpoints:")
for cp in sorted(checkpoints):
    print(f"  - {os.path.basename(cp)}")